In [2]:
import pandas as pd

In [3]:
pip install requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# get data from internet
import requests
from datetime import date

# USGS earthquake API link
api_url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# empty list to store ALL earthquakes
earthquake_list = []

# today's date
today_date = date.today()

# find date 5 years back
try:
    start_date = today_date.replace(year=today_date.year - 5)
except:
    # handle leap year issue (Feb 29)
    start_date = date(today_date.year - 5, today_date.month, 28)

print("Collecting data from", start_date, "to", today_date)

# loop through years
for year in range(start_date.year, today_date.year + 1):

    # loop through months
    for month in range(1, 13):

        # first day of month
        from_date = date(year, month, 1)

        # first day of next month
        if month == 12:
            to_date = date(year + 1, 1, 1)
        else:
            to_date = date(year, month + 1, 1)

        # skip old months (before 5 years)
        if to_date <= start_date:
            continue

        # skip future months
        if from_date > today_date:
            continue

        # adjust start date if needed
        if from_date < start_date:
            from_date = start_date

        # adjust end date if needed
        if to_date > today_date:
            to_date = today_date

        print("Fetching:", from_date, "to", to_date)

        # API parameters
        params = {
            "format": "geojson",
            "starttime": from_date.isoformat(),
            "endtime": to_date.isoformat(),
            "minmagnitude": 3
        }

        # API call
        response = requests.get(api_url, params=params)

        if response.status_code == 200:
            data = response.json()
            month_data = data.get("features", [])
            print("Records:", len(month_data))

            # add month data to main list
            earthquake_list.extend(month_data)
        else:
            print("Failed request:", response.status_code)

# total records
print("\nTotal earthquakes collected:", len(earthquake_list))


Fetching: 2021-05-18 to 2021-06-01
Records: 828
Fetching: 2021-06-01 to 2021-07-01
Records: 1613
Fetching: 2021-07-01 to 2021-08-01
Records: 1801
Fetching: 2021-08-01 to 2021-09-01
Records: 2683
Fetching: 2021-09-01 to 2021-10-01
Records: 1688
Fetching: 2021-10-01 to 2021-11-01
Records: 1536
Fetching: 2021-11-01 to 2021-12-01
Records: 1543
Fetching: 2021-12-01 to 2022-01-01
Records: 1915
Fetching: 2022-01-01 to 2022-02-01
Records: 1910
Fetching: 2022-02-01 to 2022-03-01
Records: 1615
Fetching: 2022-03-01 to 2022-04-01
Records: 1917
Fetching: 2022-04-01 to 2022-05-01
Records: 1657
Fetching: 2022-05-01 to 2022-06-01
Records: 1507
Fetching: 2022-06-01 to 2022-07-01
Records: 1530
Fetching: 2022-07-01 to 2022-08-01
Records: 1693
Fetching: 2022-08-01 to 2022-09-01
Records: 1681
Fetching: 2022-09-01 to 2022-10-01
Records: 1903
Fetching: 2022-10-01 to 2022-11-01
Records: 1618
Fetching: 2022-11-01 to 2022-12-01
Records: 1644
Fetching: 2022-12-01 to 2023-01-01
Records: 1533
Fetching: 2023-01-01 

In [5]:
import pandas as pd                                # create DataFrame from JSON data
df = pd.json_normalize(earthquake_list)
print("DataFrame created with shape:", df.shape)

DataFrame created with shape: (103609, 30)


In [6]:
df.columns

Index(['type', 'id', 'properties.mag', 'properties.place', 'properties.time',
       'properties.updated', 'properties.tz', 'properties.url',
       'properties.detail', 'properties.felt', 'properties.cdi',
       'properties.mmi', 'properties.alert', 'properties.status',
       'properties.tsunami', 'properties.sig', 'properties.net',
       'properties.code', 'properties.ids', 'properties.sources',
       'properties.types', 'properties.nst', 'properties.dmin',
       'properties.rms', 'properties.gap', 'properties.magType',
       'properties.type', 'properties.title', 'geometry.type',
       'geometry.coordinates'],
      dtype='object')

In [7]:
df.columns=['Type', 'id', 'mag', 'place', 'time',    # renaming columns for easier access
       'updated','tz','url','detail', 'felt', 'cdi',
       'mmi', 'alert', 'status',
       'tsunami', 'sig', 'net',
       'code', 'ids', 'sources',
       'types', 'nst', 'dmin',
       'rms', 'gap', 'magType',
       'type', 'title', 'gtype',
       'coordinates']

In [8]:
df["time"] = pd.to_datetime(df["time"], unit='ms')        # converting ms -> datetime
df["updated"] = pd.to_datetime(df["updated"], unit='ms')

In [9]:
df["longitude"] = df["coordinates"].apply(lambda x: x[0])   # extracting values and deriving new columns 
df["latitude"]  = df["coordinates"].apply(lambda x: x[1])      
df["depth"]  = df["coordinates"].apply(lambda x: x[2])

In [10]:
df['country'] = df['place'].str.split(',').str[-1].str.strip()   # extracting country from place description

In [11]:
df['year'] = df['time'].dt.year  # extracting year from datetime

In [12]:
df['month'] = df['time'].dt.month # extracting month from datetime

In [13]:
df['day'] = df['time'].dt.day # extracting day from datetime

In [14]:
df['day_of_week'] = df['time'].dt.day_name() # extracting day of week from datetime

In [15]:
df['mag_flag'] = df['mag'].apply(lambda x: 'destructive' if x >=6 else 'strong')  #classify depth/mag
df['depth_flag'] = df['depth'].apply(lambda x: 'shallow' if x <= 70 else 'deep')

In [16]:
df['felt'] = df['felt'].fillna(0) # filling NaN --> 0
df['cdi'] = df['cdi'].fillna(0)
df['mmi'] = df['mmi'].fillna(0)

In [17]:
df['dmin'] = df['dmin'].fillna(df['dmin'].mean()) # filling NaN with mean values
df['nst'] = df['nst'].fillna(df['nst'].mean())
df['gap'] = df['gap'].fillna(df['gap'].mean())
df['rms'] = df['rms'].fillna(df['rms'].mean())

In [18]:
df.drop(["Type","tz","url","detail","coordinates","Type","title","gtype"],axis=1,inplace=True) # drop colums

In [19]:
rearranged_column = ["id","place","country","time","updated","year","month","day","day_of_week","mag","mag_flag","magType","depth","depth_flag","longitude","latitude","type","tsunami","sources","net","alert","felt","cdi","mmi","types","sig","code","ids","nst","dmin","rms","gap","status"]
df = df[rearranged_column]

In [20]:
df # to check final dataframe

,id,place,country,time,updated,year,month,day,day_of_week,mag,...,mmi,types,sig,code,ids,nst,dmin,rms,gap,status
0,pr2021151010,"3 km NNW of La Parguera, Puerto Rico",Puerto Rico,2021-05-31 23:57:10.410,2021-08-07 22:06:47.040,2021,5,31,Monday,3.42,...,2.624,",dyfi,origin,phase-data,shakemap,",181,2021151010,",pr2021151010,us6000egfu,",23.000000,0.0471,0.27,178.0,reviewed
1,us7000eba7,"82 km SW of Adak, Alaska",Alaska,2021-05-31 23:54:58.422,2021-08-07 22:06:47.040,2021,5,31,Monday,3.40,...,0.000,",origin,phase-data,",178,7000eba7,",ak0216y462r7,us7000eba7,",45.594078,0.3980,0.36,214.0,reviewed
2,us6000egf2,"off the east coast of Honshu, Japan",Japan,2021-05-31 22:04:42.354,2025-12-22 19:01:28.495,2021,5,31,Monday,5.20,...,0.000,",internal-moment-tensor,moment-tensor,origin,p...",416,6000egf2,",usauto6000egf2,us6000egf2,iscgem620448744,",45.594078,2.3690,0.81,33.0,reviewed
3,us7000e9h9,"65 km NNW of Bukittinggi, Indonesia",Indonesia,2021-05-31 22:01:26.254,2021-08-07 22:06:46.040,2021,5,31,Monday,4.40,...,0.000,",origin,phase-data,",298,7000e9h9,",us7000e9h9,",45.594078,0.9500,0.78,145.0,reviewed
4,us7000e9h8,Guam region,Guam region,2021-05-31 21:41:30.578,2021-08-07 22:06:46.040,2021,5,31,Monday,4.40,...,0.000,",origin,phase-data,",298,7000e9h8,",us7000e9h8,",45.594078,0.3430,0.85,63.0,reviewed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103604,pr71515233,"10 km NNE of Brenas, Puerto Rico",Puerto Rico,2026-05-01 02:13:24.250,2026-05-01 02:36:25.820,2026,5,1,Friday,3.22,...,0.000,",origin,phase-data,",160,71515233,",us7000shgg,pr71515233,",41.000000,0.2318,0.20,188.0,reviewed
103605,nc75354667,"24 km W of Petrolia, CA",CA,2026-05-01 01:58:41.390,2026-05-01 05:57:21.998,2026,5,1,Friday,3.69,...,2.558,",dyfi,moment-tensor,nearby-cities,origin,phase...",210,75354667,",nc75354667,us7000shgd,",64.000000,0.1761,0.22,239.0,reviewed
103606,aka2026intneh,"115 km SSE of False Pass, Alaska",Alaska,2026-05-01 01:26:59.448,2026-05-07 01:11:31.451,2026,5,1,Friday,3.00,...,0.000,",origin,phase-data,",138,a2026intneh,",aka2026intneh,",26.000000,0.9000,0.50,247.0,reviewed
103607,pr71515218,"74 km N of Hatillo, Puerto Rico",Puerto Rico,2026-05-01 01:02:05.820,2026-05-01 01:21:41.770,2026,5,1,Friday,3.08,...,0.000,",origin,phase-data,",146,71515218,",pr71515218,",16.000000,0.7170,0.11,258.0,reviewed


In [21]:
pip install sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [25]:
import pandas as pd

df = pd.read_csv("earthquake_data.csv")

import pandas as pd  # converting into sql DB
from sqlalchemy import create_engine

engine = create_engine("mysql+pymysql://root***@localhost/capstones")

df_cleaned = df  

df_cleaned.to_sql(
    name='earthquake',
    con=engine,
    if_exists='replace',
    index=False
)

print("super")